# Hierarchy Expansion Benchmarking

This notebook demonstrates how to benchmark SNOMED CT hierarchy expansion methods using real UK SNOMED CT clinical data.

## Overview

Hierarchy expansion evaluates how well a method can find related concepts through parent-child relationships in the SNOMED CT ontology.

### Key Metrics:
- **Exact Match Rate**: Fraction of cases where all expected concepts are found exactly
- **Recall@K**: Fraction of expected concepts found in top-K results
- **Precision@K**: Fraction of top-K results that are relevant
- **F1@K**: Harmonic mean of precision and recall at K
- **Jaccard Similarity**: Overlap between predicted and expected sets

In [ ]:
import random

from snomed_methods import SnomedRelations, SnomedTermLookup

In [ ]:
relations = SnomedRelations(
    snomed_rf2_full_path="/workspaces/snomed_methods/uk_sct2cl_42.2.0/SnomedCT_UKClinicalRF2_PRODUCTION_20260603T000001Z/Full/Terminology/sct2_Relationship_UKCLFull_GB1000000_20260603.txt",
    medcat=False,
)
print(f"Loaded {len(relations.df):,} relationships")

In [ ]:
term_lookup = SnomedTermLookup(
    snomed_description_path="/workspaces/snomed_methods/uk_sct2cl_42.2.0/SnomedCT_UKClinicalRF2_PRODUCTION_20260603T000001Z/Full/Terminology/sct2_Description_UKCLFull-en_GB1000000_20260603.txt",
)
print(f"Loaded {len(term_lookup.df):,} descriptions")

## Key Concept: What This Notebook Tests

**Ground Truth** = Actual parent-child relationships from SNOMED RF2 data (using get_children/get_parents)
**Predictions** = Your expansion method's output
**Metrics** = How well predictions match ground truth

This notebook provides EXAMPLE methods using DIFFERENT algorithms than the ground truth generation to properly test expansion logic. The example methods below use:
- **Random sampling**: Selects random concepts from active set (should FAIL against ground truth)
- **Term-based semantic**: Uses string matching on concept terms
- **Same-depth siblings**: Uses parent relationship to find siblings (different algorithm than child-focused ground truth)

In [ ]:
# Example Method 1: Random sampling (should FAIL against ground truth)
def random_sampling_expansion(seed_cui, max_results=20):
    """Selects random concepts from the active concept set.
    This method does NOT use relations.get_children() or relations.get_parents().
    Should perform poorly as it has no semantic relation to ground truth.
    """
    try:
        all_concepts = (
            term_lookup.concept_ids if hasattr(term_lookup, "concept_ids") else []
        )
        if len(all_concepts) == 0:
            return []
        random.seed(hash(seed_cui) % 2**32)
        sampled = random.sample(all_concepts, min(max_results, len(all_concepts)))
        return [str(c) for c in sampled]
    except Exception:
        return []


# Example Method 2: Term-based semantic approach
def term_based_expansion(seed_cui, max_results=20):
    """Uses term matching to find related concepts.
    Retrieves the seed concept's preferred term, then finds other concepts
    with similar terms using substring matching.
    This does NOT use relations.get_children() or relations.get_parents().
    """
    try:
        cui_str = str(seed_cui)
        concept_info = term_lookup.getconcept_info(cui_str)
        if not concept_info or "preferred_name" not in concept_info:
            return []

        preferred_term = str(concept_info["preferred_name"])
        matches = term_lookup.find_concepts_by_term(
            preferred_term,
            ignore_case=True,
            match_prefix=False,
            top_n=max_results + 5,
        )
        results = [cui for cui, term in matches if str(cui) != seed_cui]
        return results[:max_results]
    except Exception:
        return []

In [ ]:
# Test the example methods
test_cui = "849871000000100"

print("Random sampling:", random_sampling_expansion(test_cui)[:5])
print("Term based:", term_based_expansion(test_cui)[:5])

In [ ]:
# Build ground truth for sampled concepts
# (Assumes dataset list exists with seed_cui entries)
def build_ground_truth(seeds):
    gt = []
    for seed in seeds:
        children = relations.get_children(int(seed))
        parents = relations.get_parents(int(seed))
        gt.append(
            {
                "seed_cui": seed,
                "expected_children": [str(c) for c in children],
                "expected_parents": [str(p) for p in parents],
            },
        )
    return gt


# Sample some concept IDs from your relations data
sample_seeds = ["849871000000100", "399267005", "415831000"]
dataset = build_ground_truth(sample_seeds)

# Benchmark different methods
# results = evaluate_hierarchy_expansion(expansion_func=random_sampling_expansion, dataset=dataset, k_values=[5, 10, 20])

print("=== To use this notebook:")
print("1. Build ground truth with build_ground_truth() for your sample concepts")
print("2. Use random_sampling_expansion(), term_based_expansion() or YOUR methods")